In [68]:
import psycopg 
import os
from dotenv import load_dotenv
import mlflow
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)




DB_DESTINATION_HOST = os.getenv("DB_DESTINATION_HOST")
DB_DESTINATION_PORT = os.getenv("DB_DESTINATION_PORT")
DB_DESTINATION_NAME = os.getenv("DB_DESTINATION_NAME")
DB_DESTINATION_USER = os.getenv("DB_DESTINATION_USER")
DB_DESTINATION_PASSWORD = os.getenv("DB_DESTINATION_PASSWORD")

load_dotenv()
connection = {
    "sslmode": "require",
    "target_session_attrs": "read-write"
}

postgres_credentials = {
    "host": DB_DESTINATION_HOST,
    "port": DB_DESTINATION_PORT,
    "dbname": DB_DESTINATION_NAME,
    "user": DB_DESTINATION_USER,
    "password": DB_DESTINATION_PASSWORD,
}

assert all(
    var_value != "" and var_value is not None
    for var_value in postgres_credentials.values()
)

connection.update(postgres_credentials)

TABLE_NAME = "users_churn"

with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df.head()

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,5768,8498-XXGWA,2014-09-01,NaT,Month-to-month,Yes,Mailed check,55.15,3673.15,DSL,...,Yes,No,No,No,Female,0,Yes,No,No,0
1,1,7590-VHVEG,2020-01-01,NaT,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,...,No,No,No,No,Female,0,Yes,No,None,0
2,5,9237-HQITU,2019-09-01,2019-11-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,...,No,No,No,No,Female,0,No,No,No,1
3,6,9305-CDSKC,2019-03-01,2019-11-01,Month-to-month,Yes,Electronic check,99.65,820.50,Fiber optic,...,Yes,No,Yes,Yes,Female,0,No,No,Yes,1
4,7,1452-KIOVK,2018-04-01,NaT,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,Fiber optic,...,No,No,Yes,No,Male,0,No,Yes,Yes,0


In [69]:
with open("columns.txt", "w", encoding="utf-8") as fio:
    fio.write(",".join(df.columns))

In [70]:
counts_columns = [
    "type", "paperless_billing", "internet_service", "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies", "gender", "senior_citizen", "partner", "dependents",
    "multiple_lines", "target"
]

stats = {}

for col in counts_columns:
    # 1. Считаем уникальные значения и преобразуем в словарь
    column_stat = df[col].value_counts().to_dict()
    column_stat = {f"{col}_{key}": value for key, value in column_stat.items()}

    # 2. Обновляем словарь stats полученными значениями
    stats.update(column_stat)


# 3. Расчет агрегированных метрик для числовых и текстовых колонок
stats["data_length"] = df.shape[0]
stats["monthly_charges_min"] = df["monthly_charges"].min()
stats["monthly_charges_max"] = df["monthly_charges"].max()          # Максимальное значение
stats["monthly_charges_mean"] = df["monthly_charges"].mean()        # Среднее значение
stats["monthly_charges_median"] = df["monthly_charges"].median()    # Медианное значение

stats["total_charges_min"] = df["total_charges"].min()              # Минимальное значение
stats["total_charges_max"] = df["total_charges"].max()              # Максимальное значение
stats["total_charges_mean"] = df["total_charges"].mean()            # Среднее значение
stats["total_charges_median"] = df["total_charges"].median()        # Медианное значение

stats["unique_customers_number"] = df["customer_id"].nunique()      # Кол-во уникальных ID
stats["end_date_nan"] = df["end_date"].isnull().sum()              # Кол-во пустых строк (NaN)


In [71]:
df.to_csv("users_churn.csv", index=False) 

In [72]:
full_data_set = df.copy()
full_data_set.head(5)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,5768,8498-XXGWA,2014-09-01,NaT,Month-to-month,Yes,Mailed check,55.15,3673.15,DSL,...,Yes,No,No,No,Female,0,Yes,No,No,0
1,1,7590-VHVEG,2020-01-01,NaT,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,...,No,No,No,No,Female,0,Yes,No,None,0
2,5,9237-HQITU,2019-09-01,2019-11-01,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,...,No,No,No,No,Female,0,No,No,No,1
3,6,9305-CDSKC,2019-03-01,2019-11-01,Month-to-month,Yes,Electronic check,99.65,820.50,Fiber optic,...,Yes,No,Yes,Yes,Female,0,No,No,Yes,1
4,7,1452-KIOVK,2018-04-01,NaT,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,Fiber optic,...,No,No,Yes,No,Male,0,No,Yes,Yes,0


In [73]:
cols_to_drop = ['begin_date', 'end_date', 'customer_id', 'gender', 'senior_citizen', 'id']
full_data_set = full_data_set.drop(columns=cols_to_drop)

display(full_data_set.head())

,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,partner,dependents,multiple_lines,target
0,Month-to-month,Yes,Mailed check,55.15,3673.15,DSL,Yes,No,Yes,No,No,No,Yes,No,No,0
1,Month-to-month,Yes,Electronic check,29.85,29.85,DSL,No,Yes,No,No,No,No,Yes,No,None,0
2,Month-to-month,Yes,Electronic check,70.70,151.65,Fiber optic,No,No,No,No,No,No,No,No,No,1
3,Month-to-month,Yes,Electronic check,99.65,820.50,Fiber optic,No,No,Yes,No,Yes,Yes,No,No,Yes,1
4,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,Fiber optic,No,Yes,No,No,Yes,No,No,Yes,Yes,0


In [74]:
TABLE_NAME = 'users_churn'# ваш код здесь

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'churn_grid_and_random_search'# ваш код здесь
RUN_NAME = 'model_grid_search' # ваш код здесь
REGISTRY_MODEL_NAME = "baseline_model_improvement"# ваш код здесь

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = "begin_date"
stratify_column = "target"
test_size = 0.25

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=test_size, shuffle=False)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

loss_function = "Logloss"
task_type = 'CPU'
random_seed = 0
iterations = 300
verbose = False

params = {
    'depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.1, 0.9],
    'iterations': [1, 2, 3],
    'l2_leaf_reg': [1, 5, 10, 15, 20],
}

model = CatBoostClassifier(
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    verbose=verbose,
)

cv = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=2,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

clf = cv.fit(X_train, y_train)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"

print(
    "AWS_ACCESS_KEY_ID:",
    "SET" if os.getenv("AWS_ACCESS_KEY_ID") else "NOT SET"
)

print(
    "AWS_SECRET_ACCESS_KEY:",
    "SET" if os.getenv("AWS_SECRET_ACCESS_KEY") else "NOT SET"
)

print(
    "MLFLOW_S3_ENDPOINT_URL:",
    os.getenv("MLFLOW_S3_ENDPOINT_URL")
)


mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

cv_results = pd.DataFrame(clf.cv_results_)

best_params = clf.best_params_

model_best = CatBoostClassifier(
    **best_params,
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    verbose=verbose
)

model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

# расчёт метрик качества
metrics = {}

_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()

auc = roc_auc_score(
    y_test,
    probas
)
precision = precision_score(
    y_test,
    prediction
)
recall = recall_score(
    y_test,
    prediction
)
f1 = f1_score(
    y_test,
    prediction
)
logloss = log_loss(y_test, prediction)

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

# дополнительные метрики из результатов кросс-валидации
metrics['mean_fit_time'] = cv_results['mean_fit_time'].mean()

metrics['std_fit_time'] = cv_results['std_fit_time'].mean()

metrics['mean_test_score'] = cv_results['mean_test_score'].mean()

metrics['std_test_score'] = cv_results['std_test_score'].mean()

metrics["best_score"] = clf.best_score_

# настройки для логирования в MLFlow

signature = mlflow.models.infer_signature(X_test, prediction)

input_example = X_test[:10]

# Получаем эксперимент по имени
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# Если эксперимента ещё нет — создаём
if experiment is None:
    experiment_id = mlflow.create_experiment(
        name=EXPERIMENT_NAME
    )
    print(f"Эксперимент создан: {EXPERIMENT_NAME}")
else:
    experiment_id = experiment.experiment_id
    print(f"Эксперимент уже существует: {EXPERIMENT_NAME}")

print(f"Experiment ID: {experiment_id}")

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    # параметры лучшей модели
    mlflow.log_params(best_params)

    # общие параметры
    mlflow.log_param("loss_function", loss_function)
    mlflow.log_param("task_type", task_type)
    mlflow.log_param("random_seed", random_seed)
    mlflow.log_param("test_size", test_size)
    mlflow.log_param("cv", 4)
    
    cv_info = mlflow.sklearn.log_model(cv, artifact_path='cv')
    
    model_info = mlflow.catboost.log_model(
        registered_model_name=REGISTRY_MODEL_NAME,        
        cb_model=model_best,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        )
    # метрики
    mlflow.log_metrics(metrics)

    # сохраняем результаты GridSearchCV
    cv_results.to_csv(
        "cv_results.csv",
        index=False
    )

    mlflow.log_artifact(
        "cv_results.csv",
        artifact_path="cv_results"
    )
    print("\nRun ID:", run_id)
    print("Experiment ID:", experiment_id)
    print("Registered model:", REGISTRY_MODEL_NAME)


Размер выборки для обучения: (5282, 3)
Размер выборки для теста: (1761, 3)
AWS_ACCESS_KEY_ID: SET
AWS_SECRET_ACCESS_KEY: SET
MLFLOW_S3_ENDPOINT_URL: https://storage.yandexcloud.net
Эксперимент уже существует: churn_grid_and_random_search
Experiment ID: 12


/home/mle-user/mle_projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'baseline_model_improvement' already exists. Creating a new version of this model...
2026/09/17 18:14:19 INFO mlflow.tracking._model_r


Run ID: d5565e352bd247e690528c731f1d9603
Experiment ID: 12
Registered model: baseline_model_improvement


Created version '3' of model 'baseline_model_improvement'.


In [ ]:
TABLE_NAME = "users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "churn_with_random_search"
RUN_NAME = "model_random_search"
REGISTRY_MODEL_NAME = "baseline_model_improvement_v2"


features = [
    "monthly_charges",
    "total_charges",
    "senior_citizen"
]

target = "target"

split_column = "begin_date"
stratify_column = "target"

test_size = 0.25


df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=test_size,
    shuffle=False
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")


loss_function = "Logloss"
task_type = "CPU"
random_seed = 0
verbose = False


param_distributions = {
    "depth": [3, 4, 5, 6, 7],
    "learning_rate": [0.01, 0.1, 0.9],
    "iterations": [1, 2, 3],
    "l2_leaf_reg": [1, 5, 10, 15, 20]
}


model = CatBoostClassifier(
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    verbose=verbose
)


cv = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=20,
    cv=2,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True,
    random_state=random_seed
)


clf = cv.fit(
    X_train,
    y_train
)

print("\nRandomizedSearchCV завершён.")

print("\nЛучшие параметры:")
print(clf.best_params_)

print("\nЛучший ROC-AUC CV:")
print(clf.best_score_)


cv_results = pd.DataFrame(
    clf.cv_results_
)

best_params = clf.best_params_


model_best = CatBoostClassifier(
    **best_params,
    loss_function=loss_function,
    task_type=task_type,
    random_seed=random_seed,
    verbose=verbose
)


model_best.fit(
    X_train,
    y_train
)


prediction = model_best.predict(
    X_test
)

probas = model_best.predict_proba(
    X_test
)[:, 1]


metrics = {}


# Confusion Matrix
tn, fp, fn, tp = confusion_matrix(
    y_test,
    prediction,
    normalize="all"
).ravel()

# Оставляем названия err1 / err2,
# если именно их ожидает задание
err1 = fp
err2 = fn


# ROC-AUC
auc = roc_auc_score(
    y_test,
    probas
)


# Precision
precision = precision_score(
    y_test,
    prediction
)


# Recall
recall = recall_score(
    y_test,
    prediction
)


# F1
f1 = f1_score(
    y_test,
    prediction
)


# LogLoss считаем по вероятностям
logloss = log_loss(
    y_test,
    probas
)


metrics["err1"] = err1
metrics["err2"] = err2

metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss


# Метрики RandomizedSearchCV
metrics["mean_fit_time"] = (
    cv_results["mean_fit_time"].mean()
)

metrics["std_fit_time"] = (
    cv_results["std_fit_time"].mean()
)

metrics["mean_test_score"] = (
    cv_results["mean_test_score"].mean()
)

metrics["std_test_score"] = (
    cv_results["std_test_score"].mean()
)

metrics["best_score"] = (
    clf.best_score_
)


print("\nМетрики лучшей модели:")

for metric_name, metric_value in metrics.items():
    print(
        f"{metric_name}: {metric_value}"
    )

os.environ["MLFLOW_S3_ENDPOINT_URL"] = (
    "https://storage.yandexcloud.net"
)

print(
    "\nAWS_ACCESS_KEY_ID:",
    "SET"
    if os.getenv("AWS_ACCESS_KEY_ID")
    else "NOT SET"
)

print(
    "AWS_SECRET_ACCESS_KEY:",
    "SET"
    if os.getenv("AWS_SECRET_ACCESS_KEY")
    else "NOT SET"
)

print(
    "MLFLOW_S3_ENDPOINT_URL:",
    os.getenv("MLFLOW_S3_ENDPOINT_URL")
)


MLFLOW_TRACKING_URI = (
    f"http://{TRACKING_SERVER_HOST}:"
    f"{TRACKING_SERVER_PORT}"
)

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI
)

mlflow.set_registry_uri(
    MLFLOW_TRACKING_URI
)

print(
    "\nTracking URI:",
    mlflow.get_tracking_uri()
)


signature = mlflow.models.infer_signature(
    X_test,
    prediction
)

input_example = X_test.iloc[:10].copy()


experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_NAME
)

if experiment is None:

    experiment_id = mlflow.create_experiment(
        name=EXPERIMENT_NAME
    )

    print(
        f"\nЭксперимент создан: "
        f"{EXPERIMENT_NAME}"
    )

else:

    experiment_id = experiment.experiment_id

    print(
        f"\nЭксперимент уже существует: "
        f"{EXPERIMENT_NAME}"
    )


print(
    "Experiment ID:",
    experiment_id
)


with mlflow.start_run(
    run_name=RUN_NAME,
    experiment_id=experiment_id
) as run:

    run_id = run.info.run_id

    print(
        "\nRun ID:",
        run_id
    )


    mlflow.log_params(
        best_params
    )


    mlflow.log_param(
        "loss_function",
        loss_function
    )

    mlflow.log_param(
        "task_type",
        task_type
    )

    mlflow.log_param(
        "random_seed",
        random_seed
    )

    mlflow.log_param(
        "test_size",
        test_size
    )

    mlflow.log_param(
        "cv",
        2
    )

    mlflow.log_param(
        "n_iter",
        20
    )


    cv_info = mlflow.sklearn.log_model(
        sk_model=cv,
        artifact_path="cv"
    )


    model_info = mlflow.catboost.log_model(
        cb_model=model_best,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example
    )


    mlflow.log_metrics(
        metrics
    )


    cv_results.to_csv(
        "cv_results_random_search.csv",
        index=False
    )

    mlflow.log_artifact(
        "cv_results_random_search.csv",
        artifact_path="cv_results"
    )


    print("\n================================")
    print("MLflow logging завершён")
    print("================================")

    print(
        "Run ID:",
        run_id
    )

    print(
        "Experiment ID:",
        experiment_id
    )

    print(
        "Experiment:",
        EXPERIMENT_NAME
    )

    print(
        "Registered model:",
        REGISTRY_MODEL_NAME
    )

    print(
        "Best params:",
        best_params
    )

    print(
        "Best CV ROC-AUC:",
        clf.best_score_
    )

    print(
        "Test ROC-AUC:",
        auc
    )

Размер выборки для обучения: (5282, 3)
Размер выборки для теста: (1761, 3)

RandomizedSearchCV завершён.

Лучшие параметры:
{'learning_rate': 0.9, 'l2_leaf_reg': 1, 'iterations': 2, 'depth': 5}

Лучший ROC-AUC CV:
0.7283049912747366

Метрики лучшей модели:
err1: 0.10618966496308915
err2: 0.2509937535491198
auc: 0.6841905280630771
precision: 0.6666666666666666
recall: 0.4583333333333333
f1: 0.5432098765432098
logloss: 0.682501863364494
mean_fit_time: 0.064951092004776
std_fit_time: 0.00923585295677185
mean_test_score: 0.6222766598833885
std_test_score: 0.05099205111581924
best_score: 0.7283049912747366

AWS_ACCESS_KEY_ID: SET
AWS_SECRET_ACCESS_KEY: SET
MLFLOW_S3_ENDPOINT_URL: https://storage.yandexcloud.net

Tracking URI: http://127.0.0.1:5000

Эксперимент создан: churn_with_random_search
Experiment ID: 13

Run ID: f718f976fd13465a8ce3cd6ff94b594f


/home/mle-user/mle_projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Successfully registered model 'baseline_model_improvement_v2'.
2026/09/17 18:14:27 INFO mlflow.tracking._model_registry.client: Waiting up to 300 secon


MLflow logging завершён
Run ID: f718f976fd13465a8ce3cd6ff94b594f
Experiment ID: 13
Experiment: churn_with_random_search
Registered model: baseline_model_improvement_v2
Best params: {'learning_rate': 0.9, 'l2_leaf_reg': 1, 'iterations': 2, 'depth': 5}
Best CV ROC-AUC: 0.7283049912747366
Test ROC-AUC: 0.6841905280630771


Created version '1' of model 'baseline_model_improvement_v2'.
